# Excel Data Quality Audit
Analyze Excel workbooks sheet-by-sheet for structural and content quality issues.

In [6]:
import pandas as pd
import numpy as np
import os
import datetime
from typing import Dict, List, Any
print('Excel Audit Setup Complete.')

Excel Audit Setup Complete.


## Section 1: Metadata & Basics
Extracting file-level information and sheet structure.

In [7]:
def inspect_excel_basics(file_path: str) -> Dict[str, Any]:
    stats = os.stat(file_path)
    xl = pd.ExcelFile(file_path)
    return {
        'size_mb': round(stats.st_size / (1024 * 1024), 4),
        'last_modified': datetime.datetime.fromtimestamp(stats.st_mtime).strftime('%Y-%m-%d %H:%M:%S'),
        'sheets': xl.sheet_names,
        'abs_path': os.path.abspath(file_path)
    }

## Section 2: Data Quality Core
Profiling nulls, types, and mathematical distributions.

In [8]:
def get_sheet_quality(df: pd.DataFrame):
    metrics = []
    for col in df.columns:
        nulls = df[col].isnull().sum()
        metrics.append({
            'Column': col,
            'Nulls': nulls,
            'Null %': f'{(nulls/len(df))*100:.2f}%' if len(df)>0 else '0%',
            'Type': str(df[col].dtype)
        })
    return pd.DataFrame(metrics)

def get_stats(df: pd.DataFrame):
    # Automatic numeric stats
    return df.describe().transpose()

## Section 3: Advanced Hygiene
Placeholders, distributions, and header consistency.

In [9]:
def detect_placeholders(df: pd.DataFrame):
    placeholders = ['n/a', 'unknown', 'none', 'null', 'nan', '?', '999']
    results = []
    for col in df.columns:
        sample = df[col].astype(str).str.lower().str.strip()
        found = {p: (sample == p).sum() for p in placeholders if (sample == p).any()}
        if found:
            results.append({'Column': col, 'Placeholders': str(found)})
    return pd.DataFrame(results)

def get_top_categories(df: pd.DataFrame):
    dist = []
    for col in df.columns:
        if df[col].nunique() < 20:
            counts = df[col].value_counts().head(3).to_dict()
            dist.append({'Column': col, 'Top Values': str(counts)})
    return pd.DataFrame(dist)

## Section 4: Audit Execution
The main orchestration for Excel auditing.

In [10]:
def full_audit_excel(file_path: str):
    info = inspect_excel_basics(file_path)
    print(f"FILE: {os.path.basename(file_path)} | Sheets: {info['sheets']}")
    
    for sheet in info['sheets']:
        print(f"\n--- AUDITING SHEET: {sheet} ---")
        df = pd.read_excel(file_path, sheet_name=sheet)
        if df.empty:
            print("Sheet is Empty.")
            continue
            
        quality = get_sheet_quality(df)
        display(quality)
        
        placeholders = detect_placeholders(df)
        if not placeholders.empty:
            print("Potential Placeholders:")
            display(placeholders)
            
        stats = get_stats(df)
        if not stats.empty:
            print("Statistical Summary:")
            display(stats)

test_file = r'd:\01.CodingProjects\DataQuality\test_files\quality_test.xlsx'
if os.path.exists(test_file):
    full_audit_excel(test_file)

FILE: quality_test.xlsx | Sheets: ['Summary', 'Details', 'Raw']

--- AUDITING SHEET: Summary ---


,Column,Nulls,Null %,Type
0,id,0,0.00%,int64
1,name,0,0.00%,object
2,score,0,0.00%,float64


Statistical Summary:


,count,mean,std,min,25%,50%,75%,max
id,3.0,2.000000,1.00000,1.0,1.50,2.0,2.50,3.0
score,3.0,84.566667,5.95511,78.2,81.85,85.5,87.75,90.0



--- AUDITING SHEET: Details ---


...

Potential Placeholders:


,Column,Placeholders
0,status,{'unknown': 1}
1,notes,"{'nan': 1, '999': 1}"


Statistical Summary:


,count,mean,std,min,25%,50%,75%,max
user_id,4.0,102.5,1.290994,101.0,101.75,102.5,103.25,104.0



--- AUDITING SHEET: Raw ---
Sheet is Empty.
